# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading, reviewing, and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` Python library. The dataset is described using the [Croissant schema standard](https://mlcommons.org/croissant), and is accessible via a public JSON-LD file.

### Dataset Source
The dataset schema is available at:

[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` and pandas are installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`. The metadata object provides access to dataset properties, including available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
### Record Sets, Fields, and their IDs
Review available record sets, their `@id` values, and the fields (`@id`) for each. This helps you understand the structure and decide which sets to explore or analyze. All entities are referenced by their `@id`.

In [ ]:
# Discover all record sets and field @id's in the dataset
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '(none)')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. Use the record set and field `@id`s identified above. 

> **Note**: Access all data through `dataset.records(record_set=<@id>)`. Field names in result rows are also the field `@id`s.

In [ ]:
# Gather all record set @ids for extraction
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Extract records as dictionaries (field @id as keys)
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id}")
        print(f"  Columns (@id): {df.columns.tolist()}")
        display(df.head())
        print("")
    else:
        print(f"[Warning] No records found for record set: {record_set_id}\n")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping or aggregating by categorical fields. 

Below is an EDA workflow for a sample numeric field and grouping by a categorical field. **Substitute the correct field `@id` values as identified above.**

In [ ]:
# Choose record set and field @ids for EDA
if record_set_ids:
    # For demonstration, pick the first loaded record set with records
    eda_record_set_id = record_set_ids[0]
    df_eda = dataframes[eda_record_set_id]
    print(f"Using record set: {eda_record_set_id}")
    
    # Attempt to auto-detect a numeric field and a categorical field
    numeric_field_id = None
    group_field_id = None
    for col in df_eda.columns:
        if pd.api.types.is_numeric_dtype(df_eda[col]):
            numeric_field_id = col
            break
    for col in df_eda.columns:
        if pd.api.types.is_string_dtype(df_eda[col]) and col != numeric_field_id:
            group_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for normalization. Skipping numeric EDA.")
    else:
        threshold = df_eda[numeric_field_id].quantile(0.5) # 50th percentile as example
        filtered_df = df_eda[df_eda[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalization (z-score)
        colnorm = f"{numeric_field_id}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, colnorm]].head())

        # Optional grouping by a categorical field
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_").reset_index()
            print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No record sets with records available for EDA.")

## 5. Visualization
You can visualize value distributions or relationships between fields. Below is an example: a histogram for the selected numeric field and a boxplot grouped by a categorical field (using field `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we have suitable fields and data
if 'filtered_df' in locals() and numeric_field_id is not None and not filtered_df.empty:
    plt.figure(figsize=(7,3))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric/categorical field for visualization or filtered_df is empty.")

## 6. Conclusion
- The dataset exposes all record sets, fields, and their `@id` using the Croissant specification.
- You can programmatically explore all fields and records using their `@id` properties.
- This workflow can be reused for any Croissant-conformant dataset by adjusting the record set and field `@id`s as needed.
- For detailed medical or scientific analytics, refer to domain-specific field documentation in the metadata.

**Next:** Try advanced analysis or export processed data. For more, see [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/).